# Qwen3 Thinking + LoRA — HFACS Classification

**To switch tasks, only edit the Parameters cell (Cell 4).**

| Task | `TASK` value | `PROMPT_YAML` value |
|---|---|---|
| classify-direct (Q1/Q2 only) | `"classify_direct"` | `"src/llm/prompts/ASRS/classify_direct_lora.yaml"` |
| extract + classify (all HFACS) | `"extract_classify"` | `"src/llm/prompts/ASRS/ASRS_extract_and_classify_lora.yaml"` |

In [ ]:
# Cell 1 — Install packages
!pip install -q peft transformers trl accelerate bitsandbytes datasets huggingface_hub scikit-learn

In [ ]:
# Cell 2 — Mount Google Drive and set up paths
import os, sys
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

DRIVE_PATH = "/content/drive/MyDrive/hfacs_qwen"
os.chdir(DRIVE_PATH)
sys.path.insert(0, "src")

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

print(f"Working directory: {os.getcwd()}")

In [ ]:
# Cell 3 — Load config
import yaml

with open("configs/finetune_qwen3_config.yaml") as f:
    config = yaml.safe_load(f)

config["lora_qwen3"]["hf_token"] = os.environ.get("HF_TOKEN", "")
print("Config loaded.")

In [ ]:
# Cell 4 — Parameters: only edit this cell to switch tasks
TASK        = "classify_direct"    # change to "extract_classify" to run the other task
PROMPT_YAML = "src/llm/prompts/ASRS/classify_direct_lora.yaml"

config["lora_qwen3"]["task"]        = TASK
config["lora_qwen3"]["prompt_yaml"] = PROMPT_YAML
print(f"Task: {TASK}")
print(f"Prompt: {PROMPT_YAML}")

In [ ]:
# Cell 5 — Build LoRA dataset (500 train / 200 test)
from llm.build_lora_dataset_qwen3 import build_lora_dataset

if TASK == "extract_classify":
    with open("configs/config.yaml") as f:
        main_cfg = yaml.safe_load(f)
    config.update(main_cfg)
    config["lora_qwen3"]["task"]        = TASK
    config["lora_qwen3"]["prompt_yaml"] = PROMPT_YAML

build_lora_dataset(config)

cfg = config["lora_qwen3"]
TRAIN_FILE = f"{cfg['output_dir']}/lora_train.jsonl"
TEST_FILE  = f"{cfg['output_dir']}/lora_test.jsonl"
print(f"Train: {TRAIN_FILE}")
print(f"Test:  {TEST_FILE}")

In [ ]:
# Cell 6 — Load Qwen3-14B with 4-bit quantization + LoRA
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

quant_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(cfg["base_model"], token=cfg["hf_token"] or None)
model = AutoModelForCausalLM.from_pretrained(
    cfg["base_model"],
    quantization_config=quant_cfg,
    device_map="auto",
    token=cfg["hf_token"] or None,
)
model.config.use_cache = False

lora_cfg = LoraConfig(
    r=cfg["lora"]["r"],
    lora_alpha=cfg["lora"]["alpha"],
    lora_dropout=cfg["lora"]["dropout"],
    target_modules=cfg["lora"]["target_modules"],
    task_type="CAUSAL_LM",
    bias="none",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

In [ ]:
# Cell 7 — Fine-tune with SFTTrainer
from datasets import load_dataset as hf_load
from trl import SFTConfig, SFTTrainer

tr = cfg["training"]
tokenizer.model_max_length = tr["max_seq_len"]  # set max length on tokenizer
train_ds = hf_load("json", data_files=TRAIN_FILE, split="train")

def format_chat(examples):
    return {
        "text": [
            tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
            for msgs in examples["messages"]
        ]
    }

train_ds = train_ds.map(format_chat, batched=True)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    args=SFTConfig(
        output_dir=cfg["checkpoint_dir"],
        dataset_text_field="text",
        num_train_epochs=tr["epochs"],
        per_device_train_batch_size=tr["batch_size"],
        gradient_accumulation_steps=tr["grad_accum"],
        learning_rate=tr["learning_rate"],
        lr_scheduler_type="cosine",
        warmup_steps=10,
        logging_steps=10,
        save_strategy="epoch",
        save_total_limit=2,
        bf16=True,
        fp16=False,
    ),
)
trainer.train()

In [ ]:
# Cell 8 — Save LoRA adapter
model.save_pretrained(cfg["adapter_save_path"])
tokenizer.save_pretrained(cfg["adapter_save_path"])
print(f"Adapter saved → {cfg['adapter_save_path']}")

In [ ]:
# Cell 9 — Evaluate on test set
import json
from sklearn.metrics import accuracy_score, classification_report

model.eval()
model.config.use_cache = True
test_ds = hf_load("json", data_files=TEST_FILE, split="train")

def predict(messages):
    prompt = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to("cuda:0") for k, v in inputs.items()}
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=150, do_sample=False)
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    json_part = text.split("</think>")[-1].strip()
    try:
        return json.loads(json_part)["Final_Class"]
    except Exception:
        return "parse_error"

y_true, y_pred = [], []
for i, row in enumerate(test_ds):
    raw  = row["messages"][-1]["content"]
    gt   = json.loads(raw.split("</think>")[-1].strip())["Final_Class"]
    pred = predict(row["messages"])
    y_true.append(gt)
    y_pred.append(pred)
    if (i + 1) % 10 == 0:
        print(f"{i+1}/200 done...")

print(f"\nAccuracy: {accuracy_score(y_true, y_pred):.3f}")
print(classification_report(y_true, y_pred))